# Forget-MI LoKU — Kaggle Hyperparameter Sweep (Giảm Df_AUC ở 6%/10%)

> 🎯 **Mục đích**: Sweep 7 configs trên Kaggle để tìm sweet-spot cho 6%/10% (giống `run_sweep.ipynb` của Colab).
> 3% đã có kết quả tốt từ `run_kaggle_loku.ipynb` → KHÔNG sweep lại 3%.

## Lý do sweep
- 6%: Df_AUC LoKU baseline vẫn cao hơn paper → cần tăng forget signal
- 10%: case khó nhất → cần combine IHL + image-FILA

## 7 Configs (giống run_sweep.ipynb)

| ID | IHL | img_scale | epochs | kappa | Hypothesis |
|---|---|---|---|---|---|
| A_ihl100 | 1.0 | 0.3 | 8 | 2.0 | IHL bump alone |
| B_img050 | 0.75 | 0.5 | 8 | 2.0 | Image-FILA bump alone |
| C_combo_moderate | 1.0 | 0.5 | 8 | 2.0 | **Combined moderate (sweet candidate)** |
| D_combo_aggressive | 1.25 | 0.5 | 8 | 2.0 | IHL aggressive + img bump |
| E_extreme | 1.5 | 0.7 | 8 | 2.0 | Most aggressive (risk over-forget) |
| F_more_epochs | 0.75 | 0.3 | 12 | 2.0 | Just train longer |
| G_less_retain | 0.75 | 0.3 | 8 | 1.0 | Less retain anchor |

## Workflow

```
Cell 1-2 : setup + auto-detect paths
Cell 3   : base helpers + SWEEP_CONFIGS + sweep helpers
Cell 4   : 🚀 Sweep 6% (7 configs × 1 seed) — ~1.5h
Cell 5   : 🚀 Sweep 10% (7 configs × 1 seed) — ~1.5h
Cell 6   : Recommend best config / forget%
Cell 7   : (Optional) Multi-seed cho config WINNING
Cell 8   : Push GitHub
```

## Setup Kaggle 1 lần

Giống `run_kaggle_loku.ipynb`:
1. Datasets: `forget-mi-data`, `forget-mi-models`
2. Secrets: `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME`
3. GPU T4 x2

## Resume

Helper tự skip configs đã chạy (kiểm CSV row có khớp hyperparams) → safe nếu Kaggle disconnect.

In [ ]:
# ====================================
# CELL 1: Setup Kaggle env (clone repo + install deps + restore CSV)
# ====================================
import os, sys, subprocess, shutil

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"
REPO_NAME = "Forget-MI-LoKU"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

os.chdir(WORK_DIR)

if not os.path.exists(REPO_DIR):
    print(f"🔽 Clone {REPO_URL}")
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    print(f"🔄 Pull latest")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)

os.chdir(REPO_DIR)
print(f"📂 CWD: {os.getcwd()}")
subprocess.run(['git', 'log', '--oneline', '-1'])

# Restore CSV (cùng key với run_kaggle_loku — share CSV để baseline rows từ run_kaggle_loku cũng skip-able)
csv_in_repo = "experiments/results_summary_loku_kaggle.csv"
csv_target = "/kaggle/working/unlearning_output/results_summary.csv"
os.makedirs(os.path.dirname(csv_target), exist_ok=True)
if os.path.exists(csv_in_repo) and not os.path.exists(csv_target):
    shutil.copy(csv_in_repo, csv_target)
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"♻️  Restored CSV từ repo: {csv_target} ({n} rows)")
elif os.path.exists(csv_target):
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"📊 CSV hiện có: {csv_target} ({n} rows)")
else:
    print(f"📊 CSV chưa có — chạy từ đầu")

import importlib.util
required = ["peft", "pydicom", "transformers", "accelerate"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print(f"📦 Thiếu: {missing}. Đang cài...")
    get_ipython().system('pip install -q pydicom scikit-image pyyaml')
    get_ipython().system('pip install -q --force-reinstall --no-deps "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"')
else:
    print("✅ Đã có đủ deps.")

import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n🔴 KHÔNG CÓ GPU — sweep treo! Bật GPU rồi rerun Cell 1.")

print("\n✅ Setup complete")


In [ ]:
# ====================================
# CELL 2: Auto-detect Kaggle dataset paths
# ====================================
import os, glob


def _find_kaggle_dataset(slug):
    direct = f'/kaggle/input/{slug}'
    if os.path.isdir(direct):
        return direct
    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')
    return candidates[0] if candidates else None


mimic_data_root = _find_kaggle_dataset('forget-mi-data')
mimic_models_root = _find_kaggle_dataset('forget-mi-models')
KAGGLE_MIMIC_DATA_ROOT = mimic_data_root
KAGGLE_MIMIC_MODELS_ROOT = mimic_models_root

print(f"📦 forget-mi-data   → {mimic_data_root or '❌ NOT FOUND'}")
print(f"📦 forget-mi-models → {mimic_models_root or '❌ NOT FOUND'}\n")

if not (mimic_data_root and mimic_models_root):
    print("❌ Thiếu Kaggle Dataset. Add: forget-mi-data + forget-mi-models")
else:
    print("✅ Datasets OK")


In [ ]:
# ====================================
# CELL 3: Base helpers + SWEEP_CONFIGS + sweep helpers
# ====================================
import os, numpy as np, pandas as pd
from datetime import datetime

CSV_PATH = "/kaggle/working/unlearning_output/results_summary.csv"

PAPER_REF = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}

# Baseline hyperparams (cho cột reference + fallback id detection)
BASELINE_PARAMS = {'ihl': 0.75, 'img_scale': 0.3, 'epochs': 8, 'kappa': 2.0}

# 7 sweep configs (same as run_sweep.ipynb)
SWEEP_CONFIGS = {
    'A_ihl100':           {'ihl': 1.0,  'img_scale': 0.3, 'epochs': 8,  'kappa': 2.0, 'note': 'IHL bump alone'},
    'B_img050':           {'ihl': 0.75, 'img_scale': 0.5, 'epochs': 8,  'kappa': 2.0, 'note': 'Image-FILA bump alone'},
    'C_combo_moderate':   {'ihl': 1.0,  'img_scale': 0.5, 'epochs': 8,  'kappa': 2.0, 'note': 'Combined moderate (sweet candidate)'},
    'D_combo_aggressive': {'ihl': 1.25, 'img_scale': 0.5, 'epochs': 8,  'kappa': 2.0, 'note': 'IHL aggressive + img bump'},
    'E_extreme':          {'ihl': 1.5,  'img_scale': 0.7, 'epochs': 8,  'kappa': 2.0, 'note': 'Most aggressive — risk over-forget'},
    'F_more_epochs':      {'ihl': 0.75, 'img_scale': 0.3, 'epochs': 12, 'kappa': 2.0, 'note': 'Baseline + epochs 8→12'},
    'G_less_retain':      {'ihl': 0.75, 'img_scale': 0.3, 'epochs': 8,  'kappa': 1.0, 'note': 'Baseline + kappa_cls_retain 2→1'},
}


def _gold_path(forget_pct):
    if forget_pct != 3 or not KAGGLE_MIMIC_MODELS_ROOT:
        return None
    p = f"{KAGGLE_MIMIC_MODELS_ROOT}/retrained_model/model_retrained_3per"
    return p if os.path.isdir(p) else None


def _sweep_seed_done(forget_pct, seed, cfg_id, cfg):
    # Check xem config (forget_pct, seed, cfg_id) đã có trong CSV chưa.
    # Preferred: dùng 'id' = sweep_<pct>per_<cfg_id>. Fallback: hyperparam match.
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    if 'forget_pct' not in df.columns or 'seed' not in df.columns:
        return False

    pct_mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
    seed_mask = (df['seed'] == seed)

    # Preferred: id match
    if 'id' in df.columns:
        target_id = f"sweep_{forget_pct}per_{cfg_id}"
        id_mask = pct_mask & seed_mask & (df['id'].astype(str) == target_id)
        if id_mask.any():
            return True

    # Fallback: hyperparam match (cho CSV cũ trước fix id column)
    sub = df[pct_mask & seed_mask]
    if sub.empty:
        return False
    for _, row in sub.iterrows():
        match = True
        for col, val in [('ihl', cfg['ihl']), ('img_subtract', cfg['img_scale']),
                         ('epochs', cfg['epochs']), ('kappa_cls_retain', cfg['kappa'])]:
            if col in row and not pd.isna(row[col]):
                try:
                    if abs(float(row[col]) - val) > 1e-3:
                        match = False
                        break
                except Exception:
                    pass
        if match:
            return True
    return False


def get_baseline_row(forget_pct):
    # Đọc baseline rows từ CSV (đã chạy ở run_kaggle_loku.ipynb).
    # Preferred: id='loku_<pct>per'. Fallback: hyperparam match.
    if not os.path.exists(CSV_PATH):
        return None
    df = pd.read_csv(CSV_PATH)
    if 'forget_pct' not in df.columns:
        return None

    pct_mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")

    if 'id' in df.columns:
        id_mask = pct_mask & (df['id'].astype(str) == f'loku_{forget_pct}per')
        sub = df[id_mask]
        if not sub.empty:
            return sub

    # Fallback hyperparam
    sub = df[pct_mask]
    bp = BASELINE_PARAMS
    keep = []
    for _, row in sub.iterrows():
        match = True
        for col, val in [('ihl', bp['ihl']), ('img_subtract', bp['img_scale']),
                         ('epochs', bp['epochs']), ('kappa_cls_retain', bp['kappa'])]:
            if col in row and not pd.isna(row[col]):
                try:
                    if abs(float(row[col]) - val) > 1e-3:
                        match = False
                        break
                except Exception:
                    pass
        if match:
            keep.append(row)
    return pd.DataFrame(keep) if keep else None


def run_sweep(forget_pct, configs=None, seeds=(42,), force_redo=False):
    # Run sweep tại forget_pct cho dict configs (default = SWEEP_CONFIGS).
    if isinstance(seeds, int):
        seeds = (seeds,)
    seeds = tuple(seeds)
    assert forget_pct in (3, 6, 10)
    if configs is None:
        configs = SWEEP_CONFIGS
    forget_csv = f"./data_splits/forget_set_{forget_pct}per.csv"
    gold_path = _gold_path(forget_pct)
    has_gold = gold_path is not None

    print(f"\n{'#'*72}")
    print(f"# 🚀 SWEEP — FORGET {forget_pct}% — {len(configs)} configs × {len(seeds)} seed(s)")
    print(f"# Gold retrained: {'✅ ' + gold_path if has_gold else '❌ N/A (1−CosSim KHÔNG hợp lệ)'}")
    print(f"{'#'*72}\n")

    if not os.path.exists(forget_csv):
        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")

    for cfg_id, cfg in configs.items():
        print(f"\n{'='*60}\n🧪 CONFIG {cfg_id} — IHL={cfg['ihl']}, img={cfg['img_scale']}, "
              f"epochs={cfg['epochs']}, kappa={cfg['kappa']}\n   Note: {cfg['note']}\n{'='*60}")
        for s in seeds:
            if not force_redo and _sweep_seed_done(forget_pct, s, cfg_id, cfg):
                print(f"⏭️  SEED {s} cho {cfg_id} đã có trong CSV — skip (force_redo=True để rerun)")
                continue
            print(f"\n🎲 SEED {s} ▸ {cfg_id} @ FORGET {forget_pct}%")
            ovr_parts = [
                f"forget_set_path={forget_csv}",
                f"id=sweep_{forget_pct}per_{cfg_id}",
                f"ihl_forget_weight={cfg['ihl']}",
                f"loku_image_subtract_scale={cfg['img_scale']}",
                f"unlearn_epochs={cfg['epochs']}",
                f"kappa_cls_retain={cfg['kappa']}",
                f"base_model_path={KAGGLE_MIMIC_MODELS_ROOT}/base_model/training_original_model",
                f"bert_pretrained_dir={KAGGLE_MIMIC_MODELS_ROOT}/base_model/training_original_model",
                f"text_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/metadata",
                f"img_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/img_data",
            ]
            if has_gold:
                ovr_parts.append(f"retrained_model_path={gold_path}")
            OVR = ",".join(ovr_parts)
            HYP = f"Sweep {cfg_id} @ {forget_pct}%: {cfg['note']}"
            exp_name = f"sweep_{forget_pct}per_{cfg_id}_seed{s}"
            cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
                   f'--config config_loku_kaggle.yaml --fresh --seed {s} '
                   f'--override "{OVR}" '
                   f'--exp {exp_name} --hypothesis "{HYP}"')
            get_ipython().system(cmd)


print("✅ Helpers + SWEEP_CONFIGS đã load.")
print(f"   CSV path: {CSV_PATH}")
print(f"   Configs : {list(SWEEP_CONFIGS.keys())}")


In [ ]:
# ====================================
# CELL 4: 🚀 SWEEP @ FORGET 6% — 7 configs × 1 seed (~1.5h)
# ====================================
RUN_SWEEP_6 = False   # ⚙️ Set True khi muốn sweep 6%

if RUN_SWEEP_6:
    run_sweep(forget_pct=6, seeds=(42,))
else:
    print("⏭️  Skip sweep 6% (RUN_SWEEP_6=False)")


In [ ]:
# ====================================
# CELL 5: 🚀 SWEEP @ FORGET 10% — 7 configs × 1 seed (~1.5h)
# ====================================
RUN_SWEEP_10 = False   # ⚙️ Set True khi muốn sweep 10%

if RUN_SWEEP_10:
    run_sweep(forget_pct=10, seeds=(42,))
else:
    print("⏭️  Skip sweep 10% (RUN_SWEEP_10=False)")


In [ ]:
# ====================================
# CELL 6: Cross-forget% recommendation — pick best config per forget%
# ====================================
# Metric: PUS = (1 − MIA_paper) × Dt_AUC (balance privacy + utility)

import os, numpy as np, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ {CSV_PATH} không tồn tại")
else:
    df_all = pd.read_csv(CSV_PATH)
    has_id_col = 'id' in df_all.columns
    print(f"📋 Total CSV rows: {len(df_all)}, has 'id' column: {has_id_col}\n")

    md_lines = [
        "# LoKU Kaggle Sweep — Recommendations",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        "Pick best config per forget% bằng **PUS = (1−MIA_paper) × Dt_AUC**.",
        "",
    ]

    recommendations = {}
    for pct in [6, 10]:
        print(f"━━━ FORGET {pct}% ━━━")
        paper = PAPER_REF[pct]
        base = get_baseline_row(pct)
        baseline_pus = None
        if base is not None and not base.empty:
            if 'MIA_paper' in base.columns and 'Dt_AUC' in base.columns:
                b_mia = base['MIA_paper'].dropna().mean()
                b_dt = base['Dt_AUC'].dropna().mean()
                if not pd.isna(b_mia) and not pd.isna(b_dt):
                    baseline_pus = (1 - b_mia) * b_dt
                    print(f"  Baseline (n={len(base)}): MIA={b_mia:.3f}, Dt_AUC={b_dt:.3f}, PUS={baseline_pus:.3f}")

        # Get sweep rows
        if has_id_col:
            sweep_mask = df_all['id'].astype(str).str.startswith(f"sweep_{pct}per_")
            sweep_rows = df_all[sweep_mask]
        else:
            # Fallback: nhận diện sweep rows không có 'id' — bỏ qua
            sweep_rows = df_all.iloc[0:0]

        if sweep_rows.empty:
            print(f"  ⏭️  Chưa có sweep rows cho {pct}%")
            md_lines.append(f"## Forget {pct}% — _(chưa chạy sweep)_\n")
            continue

        rows_md = ["| Config | IHL | img_scale | epochs | κ | MIA_paper ↓ | Df_AUC ↓ | Dt_AUC ↑ | PUS ↑ |",
                   "|---|---|---|---|---|---|---|---|---|"]
        config_pus = {}
        for cfg_id in list(SWEEP_CONFIGS.keys()):
            cfg = SWEEP_CONFIGS[cfg_id]
            target_id = f"sweep_{pct}per_{cfg_id}"
            sub = sweep_rows[sweep_rows['id'].astype(str) == target_id]
            if sub.empty:
                continue
            row = sub.iloc[-1]
            mia = float(row.get('MIA_paper', np.nan))
            df_auc = float(row.get('Df_AUC', np.nan))
            dt_auc = float(row.get('Dt_AUC', np.nan))
            pus = (1 - mia) * dt_auc if not (pd.isna(mia) or pd.isna(dt_auc)) else float('nan')
            config_pus[cfg_id] = pus
            print(f"  {cfg_id:<22} MIA={mia:.3f}  Df={df_auc:.3f}  Dt={dt_auc:.3f}  PUS={pus:.3f}")
            rows_md.append(f"| {cfg_id} | {cfg['ihl']} | {cfg['img_scale']} | {cfg['epochs']} | {cfg['kappa']} | "
                           f"{mia:.3f} | {df_auc:.3f} | {dt_auc:.3f} | {pus:.3f} |")

        # Pick winner
        if config_pus:
            valid = {k: v for k, v in config_pus.items() if not pd.isna(v)}
            if valid:
                winner = max(valid, key=valid.get)
                recommendations[pct] = winner
                print(f"  🏆 WINNER: {winner} (PUS={valid[winner]:.3f}) — paper Df_AUC={paper['Df_AUC']}")
            else:
                winner = None
        else:
            winner = None

        md_lines.append(f"## Forget {pct}%")
        md_lines.append("")
        if baseline_pus is not None:
            md_lines.append(f"**Baseline** (IHL=0.75, img=0.3): PUS = **{baseline_pus:.3f}**")
            md_lines.append("")
        md_lines.extend(rows_md)
        md_lines.append("")
        if winner:
            md_lines.append(f"🏆 **WINNER: `{winner}`** (PUS={valid[winner]:.3f}) — note: {SWEEP_CONFIGS[winner]['note']}")
        md_lines.append(f"📊 Paper Forget-MI ({pct}%): MIA={paper['MIA_paper']} | Df_AUC={paper['Df_AUC']} | Dt_AUC={paper['Dt_AUC']}")
        md_lines.append("")

    out_md = "experiments/bang_loku_kaggle_sweep_recommendations.md"
    os.makedirs("experiments", exist_ok=True)
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))
    print(f"\n💾 Saved: {out_md}")
    print(f"\n🏆 Recommendations: {recommendations}")


In [ ]:
# ====================================
# CELL 7: (Tuỳ chọn) Multi-seed cho config WINNING
# ====================================
# SỬA tên config dưới theo recommendation Cell 6 trước khi chạy.
# Mỗi config × 3 seeds ≈ 36 phút trên T4.

RUN_MULTISEED_WINNERS = False  # ⚙️ Set True sau khi Cell 6 cho recommendation

BEST_6PER = 'C_combo_moderate'      # ⚠️ SỬA theo Cell 6 output
BEST_10PER = 'D_combo_aggressive'   # ⚠️ SỬA theo Cell 6 output

if RUN_MULTISEED_WINNERS:
    if BEST_6PER in SWEEP_CONFIGS:
        print(f"🚀 Multi-seed cho {BEST_6PER} @ 6%")
        run_sweep(forget_pct=6, configs={BEST_6PER: SWEEP_CONFIGS[BEST_6PER]}, seeds=(42, 123, 7))
    if BEST_10PER in SWEEP_CONFIGS:
        print(f"\n🚀 Multi-seed cho {BEST_10PER} @ 10%")
        run_sweep(forget_pct=10, configs={BEST_10PER: SWEEP_CONFIGS[BEST_10PER]}, seeds=(42, 123, 7))
else:
    print("⏭️  Skip multi-seed winners (RUN_MULTISEED_WINNERS=False)")
    print("   Set True và sửa BEST_*PER theo recommendation Cell 6.")


In [ ]:
# ====================================
# CELL 8: Push sweep results lên GitHub (Kaggle Secrets)
# ====================================
import os, shutil

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

CSV_SRC = "/kaggle/working/unlearning_output/results_summary.csv"
CSV_DST = "experiments/results_summary_loku_kaggle.csv"
if os.path.exists(CSV_SRC):
    os.makedirs("experiments", exist_ok=True)
    shutil.copy(CSV_SRC, CSV_DST)
    n = sum(1 for _ in open(CSV_DST)) - 1
    print(f"📋 Copied CSV ({n} rows) → {CSV_DST}")

def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        return (secrets.get_secret('GITHUB_TOKEN'),
                secrets.get_secret('GIT_EMAIL'),
                secrets.get_secret('GIT_NAME'))
    except Exception as e:
        print(f"⚠️  Không load được Kaggle Secrets ({e})")
        return None, None, None

TOKEN, EMAIL, NAME = load_secrets()
if not (TOKEN and EMAIL and NAME):
    print("⚠️  Thiếu credentials — skip push")
    raise SystemExit

print("🔑 Credentials từ Kaggle Secrets ✅")
get_ipython().system(f'git config user.email "{EMAIL}"')
get_ipython().system(f'git config user.name "{NAME}"')
get_ipython().system(f'git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git')
get_ipython().system('git add experiments/ 2>/dev/null')

changes = get_ipython().getoutput('git diff --cached --name-only')
if changes and any(c.strip() for c in changes):
    print("\n📦 Files commit:")
    for f in changes:
        if f.strip():
            print(f"   - {f}")
    msg = "loku kaggle sweep: 7-config hyperparameter sweep @ 6%/10%"
    get_ipython().system(f'git commit -m "{msg}"')
    get_ipython().system(f'git push origin {BRANCH}')
    print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
else:
    print("ℹ️  Không có file mới để commit.")
